# Spark Structured Streaming Join From One Kafka Topic

This notebook demonstrates how to join two logical streams when both producers write to the same Kafka topic.

In the previous example:

- Producer A wrote to `teaching-stream-A`
- Producer B wrote to `teaching-stream-B`

In this version, both producers write to:

```text
teaching-stream-shared
```

The important requirement is that each message must include a field that identifies the source producer, such as:

```python
"stream_id": "A"
```

or:

```python
"producer_id": "A"
```

Spark reads the shared Kafka topic once, splits the records into Stream A and Stream B using `stream_id`, and then joins those two filtered streams.

## Expected Producer Message Format

Both producers should send JSON messages with the same schema.

Example message from Producer A:

```python
{
    "event_id": "...",
    "stream_id": "A",
    "number": 3,
    "event_time": "2026-05-04T03:15:10.123456"
}
```

Example message from Producer B:

```python
{
    "event_id": "...",
    "stream_id": "B",
    "number": 3,
    "event_time": "2026-05-04T03:15:13.654321"
}
```

The `stream_id` field is what allows Spark to separate records from Producer A and Producer B after reading from the same Kafka topic.

In [ ]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, from_json, to_timestamp
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

## Spark And Kafka Configuration

This version uses PySpark 3.3.0. The Spark Kafka connector version should match the Spark version.

Kafka 2.8.2 is compatible with the `spark-sql-kafka-0-10` connector. The `0-10` in the package name means Kafka 0.10 and later, not Kafka version 0.10 only.

In [ ]:
HOST_IP = "10.192.9.1"
KAFKA_BOOTSTRAP_SERVERS = f"{HOST_IP}:9092"
SHARED_TOPIC = "teaching-stream-shared"

SPARK_PACKAGES = (
    "org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {SPARK_PACKAGES} pyspark-shell"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Teaching Example - Join Two Logical Streams From One Kafka Topic")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session started.")

## Define The JSON Schema

The shared topic contains messages from both producers. Since both producers use the same schema, one schema is enough.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType(), nullable=False),
    StructField("stream_id", StringType(), nullable=False),
    StructField("number", IntegerType(), nullable=False),
    StructField("event_time", StringType(), nullable=False)
])

## Read The Shared Kafka Topic Once

Spark reads from `teaching-stream-shared` and parses every Kafka message into columns.

At this point, the stream contains records from both Producer A and Producer B.

In [ ]:
shared_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", SHARED_TOPIC)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
    .selectExpr("CAST(value AS STRING) AS json_value")
    .select(from_json(col("json_value"), event_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp", to_timestamp(col("event_time")))
    .withWatermark("event_timestamp", "30 seconds")
)

print("Shared Kafka stream created.")

## Split The Shared Stream Into Two Logical Streams

Because both producers write to the same topic, Spark must use `stream_id` to separate the records.

This does not create new Kafka topics. It only creates two filtered streaming DataFrames inside Spark.

In [ ]:
stream_a = shared_stream.filter(col("stream_id") == "A")
stream_b = shared_stream.filter(col("stream_id") == "B")

print("Logical Stream A and Stream B created from the shared topic.")

## Join The Two Logical Streams

The join condition is the same idea as before:

- the number from A equals the number from B,
- the timestamps are within 10 seconds of each other.

Because both streams came from the same Kafka topic, we also explicitly ensure that A rows are joined only with B rows. The filters above already do this, but keeping clear aliases helps students understand the logic.

In [ ]:
joined_stream = (
    stream_a.alias("a")
    .join(
        stream_b.alias("b"),
        expr("""
            a.number = b.number AND
            b.event_timestamp >= a.event_timestamp - interval 10 seconds AND
            b.event_timestamp <= a.event_timestamp + interval 10 seconds
        """),
        "inner"
    )
    .select(
        col("a.number").alias("matched_number"),
        col("a.event_id").alias("event_id_A"),
        col("b.event_id").alias("event_id_B"),
        col("a.event_timestamp").alias("event_time_A"),
        col("b.event_timestamp").alias("event_time_B")
    )
)

## Print Joined Results In Jupyter

Streaming DataFrames cannot be printed with `.show()` directly. Instead, start a streaming query and use `foreachBatch`.

Each micro-batch is passed to `print_joined_batch` as a normal batch DataFrame.

In [ ]:
def print_joined_batch(batch_df, batch_id):
    row_count = batch_df.count()

    if row_count == 0:
        print(f"Batch {batch_id}: no joined rows")
    else:
        print(f"Batch {batch_id}: {row_count} joined row(s)")
        batch_df.orderBy("event_time_A", "event_time_B").show(truncate=False)


query = (
    joined_stream.writeStream
    .foreachBatch(print_joined_batch)
    .outputMode("append")
    .option("checkpointLocation", "checkpoint_stream_join_same_topic_example")
    .start()
)

print("Same-topic streaming join started. Waiting for joined results...")
query.awaitTermination()

## Stop The Query

If you interrupt the previous cell, you can run this cell to stop the query gracefully.

In [ ]:
try:
    query.stop()
    print("Streaming query stopped.")
except NameError:
    print("No active query object found.")